In [1]:
import sys
sys.path.append("..")

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

from src.data import load_data, clean_data, split_data
from src.features import build_preprocessor

df = clean_data(load_data("../data/telco_churn.csv"))
X_train, X_test, y_train, y_test = split_data(df)

In [2]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Always-No accuracy:", round(dummy.score(X_test, y_test), 3))
print(classification_report(y_test, dummy.predict(X_test), zero_division=0))

Always-No accuracy: 0.735
              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1035
           1       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409



In [3]:
model = Pipeline([
    ("prep", build_preprocessor(X_train)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("PR-AUC: ", round(average_precision_score(y_test, proba), 3))

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC: 0.842
PR-AUC:  0.633
